# 02 — Decision Engine (V2.1 — Simple & Guarded)

**Goal**: Keep analysis **clear** and **business‑credible**.

**What this notebook does**
- Loads the SaaS snapshot and aggregates KPIs at **Region × Segment**.
- Runs **what‑if** scenarios with a **simple policy model**.
- Provides a **guarded greedy plan** under budget with realistic constraints:
  - per‑cell **uplift caps** (avoid stacking unrealistic growth)
  - **diminishing returns** when repeating the same lever on the same cell
  - **max steps** per cell/lever
  - **payback gate** (reject moves with payback beyond a target months)
- Outputs a **compact summary table** (+ optional 1 heatmap) suitable for slides.

_All comments are in English, visuals are minimal by design._


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams.update({'axes.titlesize':12,'axes.labelsize':10,'legend.fontsize':9,'figure.figsize':(9,4.8)})

fmt_eur = lambda x: f"{x:,.0f} €".replace(',', ' ')
fmt_pct = lambda x: f"{x*100:,.1f}%".replace(',', ' ')


## Load & aggregate (Region × Segment)

In [ ]:

path = Path('saas_financial_snapshot.csv')
# Note: the notebook is designed to be run where this CSV exists.
df = pd.read_csv(path)

for col in ['annual_contract_value','opening_arr','new_arr','churned_arr','reactivated_arr','expansion_arr','contraction_arr','net_arr_change']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

df['segment'] = df['segment'].fillna('SMB')
df['region']  = df['region'].fillna('Other')

cells = df.groupby(['region','segment']).agg(
    opening=('opening_arr','sum'),
    arr=('annual_contract_value','sum'),
    churn=('churned_arr','sum'),
    new=('new_arr','sum'),
    react=('reactivated_arr','sum'),
    exp=('expansion_arr','sum'),
    cont=('contraction_arr','sum'),
    accounts=('account_id','nunique')
).reset_index()

cells.head()


## CFO KPIs (Opening‑based) — Net, GRR, NRR, Quick

In [ ]:


def compute_cfo_kpis(df_agg: pd.DataFrame) -> pd.DataFrame:
    out = df_agg.copy()
    out['net'] = out['new'] + out['react'] + out['exp'] - out['churn'] - out['cont']
    out['gross_churn'] = np.where(out['opening']>0, out['churn']/out['opening'], 0.0)
    out['GRR'] = 1 - out['gross_churn']
    out['NRR'] = np.where(out['opening']>0, (out['opening'] - out['churn'] + out['exp'] - out['cont'])/out['opening'], 0.0)
    denom = (out['churn'] + out['cont']).replace(0, np.nan)
    out['Quick'] = (out['new'] + out['exp'])/denom
    out['Quick'] = out['Quick'].fillna(np.inf)
    return out

base_cells = compute_cfo_kpis(cells)
base_global = compute_cfo_kpis(cells.sum(numeric_only=True).to_frame().T)
base_global


## Guardrails & effort mapping (simple, adjustable)

- **Caps per cell**: `MAX_NEW_UPLIFT`, `MAX_REACT_UPLIFT` to avoid unrealistic stacking.
- **Diminishing returns**: geometric decay `DECAY` when repeating same move.
- **Max steps per cell/lever**: hard limit for clarity.
- **Payback gate**: reject moves with payback above `MAX_PAYBACK_MONTHS` given `AU_TO_EUR` & `GROSS_MARGIN`.

_These are deliberately simple and can be tuned._


In [ ]:

# Effort coefficients (abstract units)
DEFAULT_EFFORT = {'a_churn':1.0,'a_new':0.6,'a_react':0.4,'a_exp':0.7,'a_cont':0.8}

# Guardrails
MAX_NEW_UPLIFT   = 0.20   # +20% cap per cell
MAX_REACT_UPLIFT = 0.30   # +30% cap per cell
DECAY = 0.85              # diminishing returns factor per repeated move
MAX_STEPS_PER_CELL_LEVER = 5

# Effort -> € mapping for a simple payback test
AU_TO_EUR = 10_000.0      # 1 effort unit = 10k€ all-in cost (Sales+Mktg+CS blended)
GROSS_MARGIN = 0.80
MAX_PAYBACK_MONTHS = 18

# Elementary steps per lever (kept small to stay realistic)
STEP = {'churn_bps':50, 'new_pct':0.02, 'react_pct':0.05}


## Policy model (scopes + apply with caps)

In [ ]:


def compose_policies(*policies):
    out = {'global':{}, 'segment':{}, 'region':{}, 'cell':{}}
    def merge_scope(dst, src):
        for k,v in src.items():
            if isinstance(v, dict):
                if k not in dst: dst[k] = {}
                for sk,sv in v.items():
                    if isinstance(sv, dict):
                        dst[k][sk] = {**dst[k].get(sk, {}), **sv}
                    else:
                        dst[k][sk] = sv
            else:
                dst[k] = v
    for p in policies:
        for key in ['global','segment','region','cell']:
            if key in p:
                merge_scope(out, {key:p[key]})
    return {k:v for k,v in out.items() if v}


def apply_policy(cells_df: pd.DataFrame, policy: dict, effort_coeffs=None):
    if effort_coeffs is None:
        effort_coeffs = DEFAULT_EFFORT
    sim = cells_df.copy()
    sim[['opening','churn','new','react','exp','cont']] = sim[['opening','churn','new','react','exp','cont']].astype(float)
    sim['churn_delta']=0.0; sim['new_mult']=1.0; sim['react_mult']=1.0; sim['exp_mult']=1.0; sim['cont_delta']=0.0

    def update_row(row, lev):
        if not isinstance(lev, dict):
            return row
        if 'churn_bps' in lev: row['churn_delta'] += (lev['churn_bps']/10000.0)*row['opening']
        if 'new_pct'   in lev: row['new_mult']   *= (1.0+lev['new_pct'])
        if 'react_pct' in lev: row['react_mult'] *= (1.0+lev['react_pct'])
        if 'exp_pct'   in lev: row['exp_mult']   *= (1.0+lev['exp_pct'])
        if 'cont_bps'  in lev: row['cont_delta'] += (lev['cont_bps']/10000.0)*row['opening']
        return row

    if 'global' in policy:
        sim = sim.apply(lambda r: update_row(r, policy['global']), axis=1)
    if 'segment' in policy:
        for seg, lev in policy['segment'].items():
            m = sim['segment'].eq(seg); sim.loc[m] = sim.loc[m].apply(lambda r: update_row(r, lev), axis=1)
    if 'region' in policy:
        for reg, lev in policy['region'].items():
            m = sim['region'].eq(reg); sim.loc[m] = sim.loc[m].apply(lambda r: update_row(r, lev), axis=1)
    if 'cell' in policy:
        for (reg,seg), lev in policy['cell'].items():
            m = sim['region'].eq(reg) & sim['segment'].eq(seg); sim.loc[m] = sim.loc[m].apply(lambda r: update_row(r, lev), axis=1)

    # Apply caps (keep it simple, only on new/react)
    sim['new_mult']   = np.minimum(sim['new_mult'],   1.0 + MAX_NEW_UPLIFT)
    sim['react_mult'] = np.minimum(sim['react_mult'], 1.0 + MAX_REACT_UPLIFT)

    # Simulated flows
    sim['churn_sim'] = np.maximum(0.0, sim['churn'] - sim['churn_delta'])
    sim['new_sim']   = np.maximum(0.0, sim['new'] * sim['new_mult'])
    sim['react_sim'] = np.maximum(0.0, sim['react'] * sim['react_mult'])
    sim['exp_sim']   = np.maximum(0.0, sim['exp'] * sim['exp_mult'])
    sim['cont_sim']  = np.maximum(0.0, sim['cont'] - sim['cont_delta'])

    # Effort (abstract units)
    e = effort_coeffs; effort = 0.0
    effort += (e['a_churn']*((sim['churn_delta']/sim['opening'].replace(0,np.nan)).fillna(0.0))*sim['accounts']).sum()
    effort += (e['a_new']*(sim['new_mult']-1.0)*sim['accounts']).sum()
    effort += (e['a_react']*(sim['react_mult']-1.0)*sim['accounts']).sum()
    effort += (e['a_exp']*(sim['exp_mult']-1.0)*sim['accounts']).sum()
    effort += (e['a_cont']*((sim['cont_delta']/sim['opening'].replace(0,np.nan)).fillna(0.0))*sim['accounts']).sum()

    out = sim[['region','segment','opening']].copy()
    out = out.assign(churn=sim['churn_sim'], new=sim['new_sim'], react=sim['react_sim'], exp=sim['exp_sim'], cont=sim['cont_sim'], accounts=sim['accounts'])
    return out, float(effort)


## Scenario runner & compact comparison

In [ ]:


def run_scenario(cells_df: pd.DataFrame, policy: dict, name: str, effort_coeffs=None):
    sim_cells, effort = apply_policy(cells_df, policy, effort_coeffs)
    base_g = compute_cfo_kpis(cells_df.sum(numeric_only=True).to_frame().T)
    sim_g  = compute_cfo_kpis(sim_cells.sum(numeric_only=True).to_frame().T)
    delta_g = pd.DataFrame({
        'Scenario': [name],
        'Δ Net ARR (€)': sim_g['net'].iat[0] - base_g['net'].iat[0],
        'Δ GRR (pp)': (sim_g['GRR'].iat[0] - base_g['GRR'].iat[0]) * 100,
        'Δ NRR (pp)': (sim_g['NRR'].iat[0] - base_g['NRR'].iat[0]) * 100,
        'Effort (au)': effort
    })
    return sim_cells, delta_g


def compare_scenarios(cells_df, scenarios: dict):
    rows = []
    for name, pol in scenarios.items():
        _, d = run_scenario(cells_df, pol, name)
        rows.append(d.iloc[0])
    out = pd.DataFrame(rows)
    cols = ['Scenario','Δ Net ARR (€)','Δ GRR (pp)','Δ NRR (pp)','Effort (au)']
    return out[cols]


## Optional: single heatmap (normalized Δ Net by Region)

In [ ]:


def heatmap_delta(comp_cells: pd.DataFrame, title: str):
    pvt = comp_cells.pivot(index='region', columns='segment', values='delta_net_pct_region').fillna(0.0)
    fig, ax = plt.subplots(figsize=(9,3.6))
    sns.heatmap(pvt, annot=True, fmt='.1%', cmap='YlGnBu', cbar=True, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Segment'); ax.set_ylabel('Region')
    plt.tight_layout(); plt.show()


## Greedy plan (guarded): decay, caps, max steps, payback gate

In [ ]:

from collections import defaultdict

applied_counts = defaultdict(int)


def payback_ok(delta_net_eur, effort_au):
    # gross profit proxy
    gp = max(0.0, delta_net_eur * GROSS_MARGIN)
    cost = effort_au * AU_TO_EUR
    if gp <= 0:
        return False
    payback_mo = cost / (gp / 12.0)
    return payback_mo <= MAX_PAYBACK_MONTHS


def best_move_given_state(sim_cells: pd.DataFrame, effort_coeffs=None):
    if effort_coeffs is None:
        effort_coeffs = DEFAULT_EFFORT
    base_g = compute_cfo_kpis(sim_cells.sum(numeric_only=True).to_frame().T)
    base_net = base_g['net'].iat[0]

    candidates = []
    for _, row in sim_cells.iterrows():
        reg, seg = row['region'], row['segment']
        for lever, step in STEP.items():
            n = applied_counts[(reg, seg, lever)]
            if n >= MAX_STEPS_PER_CELL_LEVER:
                continue
            step_eff = step * (DECAY ** n)
            if step_eff <= 0:
                continue
            pol = {'cell': {(reg, seg): {lever: step_eff}}}
            after, effort = apply_policy(sim_cells, pol, effort_coeffs)
            after_g = compute_cfo_kpis(after.sum(numeric_only=True).to_frame().T)
            dnet = after_g['net'].iat[0] - base_net
            # Payback test (reject if outside)
            if not payback_ok(dnet, effort):
                continue
            roi = dnet / max(effort, 1e-6)
            candidates.append({'region':reg,'segment':seg,'lever':lever,'step':step_eff,'delta_net':dnet,'effort':effort,'ROI':roi})

    if not candidates:
        return None
    cand_df = pd.DataFrame(candidates).sort_values('ROI', ascending=False)
    return cand_df.iloc[0].to_dict()


def greedy_allocate(cells_df: pd.DataFrame, budget: float, effort_coeffs=None):
    if effort_coeffs is None:
        effort_coeffs = DEFAULT_EFFORT
    sim_cells = cells_df.copy()
    spent = 0.0
    picks = []

    while spent < budget:
        move = best_move_given_state(sim_cells, effort_coeffs)
        if move is None:
            break
        if spent + move['effort'] > budget:
            break
        pol = {'cell': {(move['region'], move['segment']): {move['lever']: move['step']}}}
        after, _ = apply_policy(sim_cells, pol, effort_coeffs)
        sim_cells = after
        spent += move['effort']
        picks.append(move)
        applied_counts[(move['region'], move['segment'], move['lever'])] += 1

    base_g = compute_cfo_kpis(cells_df.sum(numeric_only=True).to_frame().T)
    final_g = compute_cfo_kpis(sim_cells.sum(numeric_only=True).to_frame().T)
    summary = pd.DataFrame([{
        'Budget used (au)': spent,
        'Δ Net ARR (€)': final_g['net'].iat[0] - base_g['net'].iat[0],
        'Δ GRR (pp)': (final_g['GRR'].iat[0] - base_g['GRR'].iat[0]) * 100,
        'Δ NRR (pp)': (final_g['NRR'].iat[0] - base_g['NRR'].iat[0]) * 100,
    }])
    return sim_cells, pd.DataFrame(picks), summary


## Examples — simple scenarios + greedy (compact outputs)

In [ ]:

# Policies kept simple (clear reading)
policy_retention = {'segment': {'Enterprise': {'churn_bps': 150}},
                    'cell':    {('EMEA','Mid-Market'): {'churn_bps': 150}}}
policy_growth_balanced = {'region': {'EMEA': {'new_pct': 0.06}},
                          'segment': {'SMB': {'new_pct': 0.04, 'react_pct': 0.10}}}
policy_protect_whales = {'cell': {('APAC','Enterprise'): {'churn_bps': 100, 'react_pct': 0.05}}}

scenarios = {
    'Retention-first': policy_retention,
    'Balanced Growth': policy_growth_balanced,
    'Protect Whales':  policy_protect_whales,
}

comp_df = compare_scenarios(cells, scenarios)
comp_df


In [ ]:

# Greedy under budget with guardrails
BUDGET = 50.0
sim_after, picks_df, summary_df = greedy_allocate(cells, budget=BUDGET)
summary_df


In [ ]:

# Optional: one heatmap for the greedy plan (normalized by region ARR)
base_eval = compute_cfo_kpis(cells)
final_eval = compute_cfo_kpis(sim_after)

reg_arr = base_eval.groupby('region')['arr'].sum().rename('arr_region')
comp = base_eval[['region','segment','arr','net']].merge(
    final_eval[['region','segment','net']], on=['region','segment'], suffixes=('_base','_sim')
).merge(reg_arr, on='region', how='left')
comp['delta_net'] = comp['net_sim'] - comp['net_base']
comp['delta_net_pct_region'] = np.where(comp['arr_region']>0, comp['delta_net']/comp['arr_region'], 0.0)

heatmap_delta(comp, 'Greedy — Normalized Δ Net (Region × Segment)')


## Executive summary (print) — keep it simple

In [ ]:

print('BASELINE (global):')
print(base_global[['opening','GRR','NRR','Quick','net']])
print('
SCENARIOS (compact):')
print(comp_df)
print('
GREEDY SUMMARY:')
print(summary_df)
if not picks_df.empty:
    print('
Top 5 greedy moves:')
    print(picks_df.sort_values('ROI', ascending=False).head(5))
else:
    print('No greedy moves selected under guardrails & budget.')


## Save

In [ ]:

out_ipynb = Path('02_decision_engine_v2_1_simple.ipynb')
with out_ipynb.open('w', encoding='utf-8') as f:
    nbf.write(nb, f)

# Optional HTML export (best-effort; ignore if nbconvert missing)
try:
    from nbconvert import HTMLExporter
    body, _ = HTMLExporter().from_notebook_node(nb)
    Path('02_decision_engine_v2_1_simple.html').write_text(body, encoding='utf-8')
    html_ok = True
except Exception:
    html_ok = False

(out_ipynb.name, '02_decision_engine_v2_1_simple.html' if html_ok else None)
